# Sign → Text — Retraining (v2)

Position/scale-invariant landmark features + augmentation + BiGRU. The feature
code in the **SHARED** cell below is byte-identical to `app/core/sign_features.py`
in the app. **Never edit one without the other** — that is the parity contract.

Expected data layout on Drive: `recordings/<GLOSS>/<clip>.mp4` (one gloss per clip).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install mediapipe opencv-python scikit-learn
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import os, json, numpy as np, tensorflow as tf
from tqdm import tqdm

DATASET_PATH = '/content/drive/MyDrive/recordings'   # recordings/<GLOSS>/<clip>.mp4
TASK_PATH    = 'hand_landmarker.task'
OUT_DIR      = '/content/drive/MyDrive/HAND_MODEL'
os.makedirs(OUT_DIR, exist_ok=True)

AUG_PER_SAMPLE = 12
VAL_SPLIT      = 0.15
EPOCHS         = 120
BATCH_SIZE     = 32
SEED           = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

## SHARED feature code — IDENTICAL to `app/core/sign_features.py`

In [ ]:
"""
sign_features.py  —  SHARED feature extraction for Sign->Text.

  *** PARITY CONTRACT ***
  This exact file is used BOTH in training (Colab notebook) and in the live app.
  If you change anything here, you MUST retrain and ship the same file to the app.
  Do not "improve" one side only — that is what broke accuracy before.

Pipeline (identical on both sides):
  1. Read video / camera frames as RGB, full resolution.
  2. Run MediaPipe HandLandmarker in IMAGE mode (frame-independent -> sampling
     density doesn't matter, so train and live always agree).
  3. Order hands deterministically by handedness: slot 0 = Left, slot 1 = Right.
     Missing hand -> that 63-float block stays zero.
  4. Uniformly sample SEQ_LEN frames across the whole single-gloss clip/segment.
  5. Normalize the WHOLE sequence: subtract the clip's mean hand anchor and divide
     by mean hand size -> position- and scale-invariant, motion preserved.
"""

import numpy as np

try:
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
    _MP_OK = True
except Exception:
    _MP_OK = False

SEQ_LEN = 30
HANDS = 2
LM = 21
C = 3
FEATURES = HANDS * LM * C  # 126

WRIST = 0
MIDDLE_MCP = 9


# ---------------------------------------------------------------------------
# MediaPipe landmarker (IMAGE mode = frame-independent = train/live parity)
# ---------------------------------------------------------------------------
def make_landmarker(task_path: str):
    base = mp_python.BaseOptions(model_asset_path=task_path)
    opts = vision.HandLandmarkerOptions(
        base_options=base,
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
    )
    return vision.HandLandmarker.create_from_options(opts)


def detect_image(landmarker, rgb_frame):
    """Run detection on one RGB frame (full resolution). Returns the result."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return landmarker.detect(mp_image)


# ---------------------------------------------------------------------------
# Result -> (kp[126], present[2])  ordered by handedness
# ---------------------------------------------------------------------------
def landmarks_from_result(result):
    kp = np.zeros(FEATURES, dtype=np.float32)
    present = np.zeros(HANDS, dtype=np.float32)
    hands = getattr(result, "hand_landmarks", None)
    if not hands:
        return kp, present

    n = min(2, len(hands))
    slot = [None, None]  # [Left, Right]
    handed = getattr(result, "handedness", None)
    for i in range(n):
        cat = None
        try:
            cat = handed[i][0].category_name
        except Exception:
            cat = None
        if cat == "Left" and slot[0] is None:
            slot[0] = i
        elif cat == "Right" and slot[1] is None:
            slot[1] = i

    used = {x for x in slot if x is not None}
    remaining = [i for i in range(n) if i not in used]
    for s in range(2):
        if slot[s] is None and remaining:
            slot[s] = remaining.pop(0)

    for s in range(2):
        i = slot[s]
        if i is None:
            continue
        present[s] = 1.0
        for j, lm in enumerate(hands[i]):
            base = s * 63 + j * 3
            kp[base:base + 3] = (lm.x, lm.y, lm.z)
    return kp, present


# ---------------------------------------------------------------------------
# Sequence normalization (position + scale invariant, motion preserved)
# ---------------------------------------------------------------------------
def normalize_sequence(seq, present):
    """seq: (T, 126) raw handedness-ordered. present: (T, 2). Returns (T, 126)."""
    seq = np.asarray(seq, dtype=np.float32).copy()
    T = seq.shape[0]
    pts = seq.reshape(T, HANDS, LM, C)
    pres = np.asarray(present, dtype=np.float32).reshape(T, HANDS) > 0.5

    anchors = []
    sizes = []
    for t in range(T):
        ws, ss = [], []
        for h in range(HANDS):
            if pres[t, h]:
                wrist = pts[t, h, WRIST]
                mcp = pts[t, h, MIDDLE_MCP]
                ws.append(wrist)
                ss.append(np.linalg.norm(mcp - wrist))
        if ws:
            anchors.append(np.mean(ws, axis=0))
            sizes.append(np.mean(ss))

    if not anchors:
        return np.zeros((T, FEATURES), dtype=np.float32)

    origin = np.mean(anchors, axis=0)
    scale = float(np.mean(sizes))
    if scale < 1e-6:
        scale = 1e-6

    out = np.zeros_like(pts)
    for t in range(T):
        for h in range(HANDS):
            if pres[t, h]:
                out[t, h] = (pts[t, h] - origin) / scale
    return out.reshape(T, FEATURES).astype(np.float32)


def sample_indices(n_frames: int, k: int = SEQ_LEN):
    return np.linspace(0, max(n_frames - 1, 0), k).astype(int)


# ---------------------------------------------------------------------------
# Whole-video -> normalized features (used in training)
# ---------------------------------------------------------------------------
def raw_sequence_from_video(video_path: str, task_path: str):
    """Returns (seq[SEQ_LEN,126], present[SEQ_LEN,2]) or (None, None) if empty."""
    landmarker = make_landmarker(task_path)
    cap = cv2.VideoCapture(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        landmarker.close()
        return None, None

    idxs = sample_indices(len(frames))
    seq = np.zeros((SEQ_LEN, FEATURES), dtype=np.float32)
    pres = np.zeros((SEQ_LEN, HANDS), dtype=np.float32)
    for r, ix in enumerate(idxs):
        res = detect_image(landmarker, frames[ix])
        kp, pr = landmarks_from_result(res)
        seq[r] = kp
        pres[r] = pr
    landmarker.close()
    return seq, pres


def features_from_video(video_path: str, task_path: str):
    """Full pipeline: video -> normalized (SEQ_LEN, 126). None if no frames."""
    seq, pres = raw_sequence_from_video(video_path, task_path)
    if seq is None:
        return None
    return normalize_sequence(seq, pres)

## Landmark augmentation
Applied to **normalized** features, training set only: small rotation, jitter,
scale wobble, left/right mirror (with hand-slot swap), and time-warp.

In [ ]:
def _slot_present(pts):
    return np.abs(pts).sum(axis=(2,3)) > 0

def aug_rotate(x, max_deg=15.0):
    th = np.deg2rad(np.random.uniform(-max_deg, max_deg))
    c, s = np.cos(th), np.sin(th)
    p = x.reshape(-1,2,21,3).copy()
    xx, yy = p[...,0].copy(), p[...,1].copy()
    p[...,0] = xx*c - yy*s
    p[...,1] = xx*s + yy*c
    return p.reshape(x.shape)

def aug_scale(x, lo=0.9, hi=1.1):
    return (x * np.random.uniform(lo, hi)).astype(np.float32)

def aug_jitter(x, sigma=0.02):
    p = x.reshape(-1,2,21,3).copy()
    m = _slot_present(p)[...,None,None]
    p = p + np.random.normal(0, sigma, p.shape) * m
    return p.reshape(x.shape).astype(np.float32)

def aug_mirror(x):
    p = x.reshape(-1,2,21,3).copy()
    p[...,0] *= -1.0
    p = p[:, [1,0], :, :]
    return p.reshape(x.shape).astype(np.float32)

def aug_timewarp(x):
    T = x.shape[0]
    a = int(np.random.uniform(0, 0.2*T)); b = int(np.random.uniform(0.8*T, T-1))
    if b <= a: a, b = 0, T-1
    idx = np.clip(np.round(np.linspace(a, b, T)), 0, T-1).astype(int)
    return x[idx].astype(np.float32)

def augment(x):
    if np.random.rand() < 0.5: x = aug_timewarp(x)
    if np.random.rand() < 0.7: x = aug_rotate(x)
    if np.random.rand() < 0.5: x = aug_scale(x)
    if np.random.rand() < 0.5: x = aug_mirror(x)
    if np.random.rand() < 0.7: x = aug_jitter(x)
    return x.astype(np.float32)

## Extract features for every clip (normalized)

In [ ]:
labels = sorted(d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d)))
label_map = {lab: i for i, lab in enumerate(labels)}
print('Classes:', labels)

X, y = [], []
for lab in labels:
    cdir = os.path.join(DATASET_PATH, lab)
    vids = [v for v in sorted(os.listdir(cdir)) if v.lower().endswith(('.mp4','.mov','.avi','.webm'))]
    for v in tqdm(vids, desc=lab):
        feat = features_from_video(os.path.join(cdir, v), TASK_PATH)
        if feat is None:
            print('  skip empty:', v); continue
        X.append(feat); y.append(label_map[lab])

X = np.array(X, dtype=np.float32); y = np.array(y)
print('X', X.shape, 'y', y.shape)
np.save(os.path.join(OUT_DIR,'X.npy'), X); np.save(os.path.join(OUT_DIR,'y.npy'), y)

## Split, then augment the training set only

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED)

aug_X, aug_y = [], []
for xi, yi in zip(X_tr, y_tr):
    aug_X.append(xi); aug_y.append(yi)
    for _ in range(AUG_PER_SAMPLE):
        aug_X.append(augment(xi)); aug_y.append(yi)
X_tr_a = np.array(aug_X, dtype=np.float32); y_tr_a = np.array(aug_y)
perm = np.random.permutation(len(X_tr_a)); X_tr_a, y_tr_a = X_tr_a[perm], y_tr_a[perm]
print('Train (augmented):', X_tr_a.shape, ' Val:', X_val.shape)

## Model — Conv1D + BiGRU

In [ ]:
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

T, F = X.shape[1], X.shape[2]; N = len(labels)

def build_model():
    inp = layers.Input((T, F))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(layers.GRU(64, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.GRU(64, dropout=0.2))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(N, activation='softmax')(x)
    m = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

model = build_model(); model.summary()
cw = compute_class_weight('balanced', classes=np.arange(N), y=y_tr)
class_weight = {i: float(w) for i, w in enumerate(cw)}

## Train

In [ ]:
cbs = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5),
]
hist = model.fit(X_tr_a, y_tr_a, validation_data=(X_val, y_val),
                 epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weight, callbacks=cbs)
print('Best val accuracy:', max(hist.history['val_accuracy']))

## Evaluate — report + confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

pred = model.predict(X_val).argmax(1)
print(classification_report(y_val, pred, target_names=labels, zero_division=0))

cm = confusion_matrix(y_val, pred, labels=np.arange(N))
plt.figure(figsize=(9,8)); plt.imshow(cm, cmap='Blues')
plt.xticks(range(N), labels, rotation=90); plt.yticks(range(N), labels)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix'); plt.colorbar()
for i in range(N):
    for j in range(N):
        if cm[i,j]: plt.text(j, i, cm[i,j], ha='center', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## Quick live check (records 2s, predicts with the SAME pipeline)

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
import base64

def record_video(seconds=2, filename='live.mp4'):
    js = Javascript('''
      async function rec(sec){
        const s = await navigator.mediaDevices.getUserMedia({video:true});
        const mr = new MediaRecorder(s); let d=[];
        mr.ondataavailable = e => d.push(e.data); mr.start();
        await new Promise(r=>setTimeout(r, sec*1000)); mr.stop();
        await new Promise(r=>mr.onstop=r);
        const b = new Blob(d,{type:'video/mp4'}); const rd = new FileReader();
        rd.readAsDataURL(b); await new Promise(r=>rd.onloadend=r); return rd.result;
      }''')
    display(js)
    data = eval_js(f'rec({seconds})')
    open(filename,'wb').write(base64.b64decode(data.split(',')[1]))
    return filename

inv = {v:k for k,v in label_map.items()}
p = record_video(2)
feat = features_from_video(p, TASK_PATH)
probs = model.predict(feat[None,...])[0]
print('Prediction:', inv[int(probs.argmax())], 'conf', float(probs.max()))

## Export model + label_map + config
Download these three and drop them into the project root (replacing the old files).
The app already ships the matching `sign_features.py`, so nothing else changes.

In [ ]:
model.save('/content/model.keras')
json.dump(label_map, open('/content/label_map.json','w'))

config = {
    'sequence_length': int(T), 'features': int(F), 'hands': 2,
    'extractor': 'MediaPipe HandLandmarker (IMAGE mode)',
    'cam_index': 0, 'cam_backend': 'auto', 'feature_normalize': 'sequence',
    'capture_mode': 'segment', 'seg_start_frames': 3, 'seg_end_frames': 8,
    'seg_min_frames': 8, 'seg_max_seconds': 5.0, 'seg_trim_frames': 0,
    'min_confidence': 0.6, 'take_seconds': 2.0,
    'gemini_enabled': True, 'gemini_api_key': '', 'gemini_model': 'gemini-2.5-flash',
    'gemini_flush_after': 2, 'gemini_persist': True, 'gemini_timeout': 8.0
}
json.dump(config, open('/content/config.json','w'), indent=2)
model.save(os.path.join(OUT_DIR,'model.keras'))

from google.colab import files
files.download('/content/model.keras')
files.download('/content/label_map.json')
files.download('/content/config.json')